1. Load DataSet

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SparkAssignment") \
    .getOrCreate()

df = spark.read.csv(
    "spark_sample_dataset.csv",
    header=True,
    inferSchema=True
)

df.show(5)
df.printSchema()

+-------+----------------+-----------+------+--------+----------------+-----------+------+---+------------+-----------------+--------+-------------------+-------+
|user_id|transaction_date|       city|region|store_id|product_category|sale_amount| price|age|subscription|            email|username|      raw_timestamp| status|
+-------+----------------+-----------+------+--------+----------------+-----------+------+---+------------+-----------------+--------+-------------------+-------+
|   1014|      2024-06-12|   New York| North|       4|        Clothing|     148.14|371.63| 52|     Premium|user1@example.com|    NULL|2024-06-12 00:00:00|   NULL|
|   1064|      2024-02-29|    Phoenix|  West|       9|        Clothing|     718.86|212.66| 46|       Basic|user2@example.com|    NULL|2024-02-29 00:00:00|Pending|
|   1035|      2024-03-28|Los Angeles|  East|       6|     Electronics|     101.82|  NULL| 40|       Basic|user3@example.com|   user3|2024-03-28 00:00:00|Pending|
|   1070|      2024-01

2. Remove Duplicates

In [2]:
df = df.dropDuplicates()

3. Fill Null Prices

In [3]:
df = df.na.fill({"price": 0})

4. Fill Null Status

In [4]:
df = df.na.fill({"status": "Unknown"})

5. Remove Invalid Rows

In [5]:
from pyspark.sql.functions import col

df = df.filter(
    col("email").isNotNull() &
    (col("username") != "")
)

In [6]:
from pyspark.sql.functions import col
from pyspark.sql.types import TimestampType

df = df.withColumn(
    "event_time",
    col("raw_timestamp").cast(TimestampType())
).drop("raw_timestamp")

Premium Users

In [7]:
premium_users = df.filter(
    (df.age.between(18,30)) &
    (df.subscription=="Premium")
)

6. Average Sales by Category in West Region

In [8]:
from pyspark.sql.functions import avg

west_sales = (
    df.filter(df.region=="West")
      .groupBy("product_category")
      .agg(avg("sale_amount").alias("avg_sale"))
)

west_sales.show()

+----------------+-----------------+
|product_category|         avg_sale|
+----------------+-----------------+
|       Groceries|708.2636363636365|
|     Electronics|           588.87|
|        Clothing|541.2223076923078|
|       Furniture|348.8140000000001|
+----------------+-----------------+



7. Count Records per city

In [9]:
from pyspark.sql.functions import count

city_counts = (
    df.groupBy("city")
      .agg(count("*").alias("total_records"))
)

city_counts.show()

+-----------+-------------+
|       city|total_records|
+-----------+-------------+
|    Phoenix|           35|
|Los Angeles|           36|
|    Chicago|           45|
|    Houston|           28|
|   New York|           37|
+-----------+-------------+



8. MIN, MAX, AVG Prices

In [10]:
from pyspark.sql.functions import min,max,mean

df.agg(
    min("price").alias("min_price"),
    max("price").alias("max_price"),
    mean("price").alias("avg_price")
).show()

+---------+---------+------------------+
|min_price|max_price|         avg_price|
+---------+---------+------------------+
|      0.0|   499.07|209.41790055248623|
+---------+---------+------------------+



9. Final Pipeline

In [11]:
from pyspark.sql.functions import sum

final_result = (
    df.dropDuplicates()
      .na.fill({"price":0})
      .groupBy("store_id")
      .agg(sum("price").alias("total_revenue"))
)

final_result.show()

+--------+------------------+
|store_id|     total_revenue|
+--------+------------------+
|       1|           4194.49|
|       6|3119.0399999999995|
|       3|3376.3199999999997|
|       5|1691.3200000000002|
|       9|           5536.92|
|       4|           3363.75|
|       8| 6228.050000000001|
|       7|           3117.69|
|      10|3850.2800000000007|
|       2| 3426.779999999999|
+--------+------------------+



In [12]:
df.show()

+-------+----------------+-----------+------+--------+----------------+-----------+------+---+------------+-------------------+--------+-------+-------------------+
|user_id|transaction_date|       city|region|store_id|product_category|sale_amount| price|age|subscription|              email|username| status|         event_time|
+-------+----------------+-----------+------+--------+----------------+-----------+------+---+------------+-------------------+--------+-------+-------------------+
|   1072|      2024-06-06|    Chicago| South|       9|     Electronics|     310.84|393.95| 56|       Basic| user30@example.com|  user30|Pending|2024-06-06 00:00:00|
|   1008|      2024-02-28|   New York| North|       2|        Clothing|     285.69|271.92| 54|       Basic| user13@example.com|  user13| Active|2024-02-28 00:00:00|
|   1047|      2024-06-24|    Phoenix| North|       1|       Furniture|     283.12|474.76| 47|     Premium| user75@example.com|  user75| Active|2024-06-24 00:00:00|
|   1011| 